Total de ventas por mes y año con porcentaje de crecimiento respecto del mes anterior

In [0]:
WITH VentasMensuales AS (
  SELECT
    d.year,
    d.month,
    SUM(f.sale_dollars) AS total_sales
  FROM iowa_sales.sales_gold.fact_sales f
  JOIN iowa_sales.sales_gold.dim_date d ON f.date_key = d.date_key
  GROUP BY d.year, d.month
)
SELECT
  year,
  month,
  total_sales,
  LAG(total_sales) OVER (ORDER BY year, month) AS prev_month_sales,
  ROUND(((total_sales - LAG(total_sales) OVER (ORDER BY year, month)) / LAG(total_sales) OVER (ORDER BY year, month)) * 100, 2) AS growth_pct
FROM VentasMensuales
ORDER BY year DESC, month DESC;

year,month,total_sales,prev_month_sales,growth_pct
2025,12,32215643.31,24718856.70,30.33
2025,11,24718856.70,27650419.75,-10.60
2025,10,27650419.75,26289362.89,5.18
2025,9,26289362.89,25776116.31,1.99
2025,8,25776116.31,26756494.00,-3.66
2025,7,26756494.00,27876507.76,-4.02
2025,6,27876507.76,26586021.72,4.85
2025,5,26586021.72,26918526.27,-1.24
2025,4,26918526.27,25122742.04,7.15
2025,3,25122742.04,22537999.88,11.47


Qué tiendas y en qué condados se mueve más volumen de litros físicos, además del dinero recaudado.

In [0]:
SELECT
  s.store_name,
  s.county,
  SUM(f.volume_sold_liters) AS total_volume_liters,
  SUM(f.sale_dollars) AS total_sales_dollars
FROM iowa_sales.sales_gold.fact_sales f
JOIN iowa_sales.sales_gold.dim_store s ON f.store_key = s.store_key
GROUP BY s.store_name, s.county
ORDER BY total_volume_liters DESC
LIMIT 10;

store_name,county,total_volume_liters,total_sales_dollars
CENTRAL CITY 2,POLK,1164469.888,60444943.70
HY-VEE #3 / BDI / DES MOINES,POLK,1002485.132,59068425.44
HY-VEE WINE AND SPIRITS #1 (1281) / IOWA CITY,JOHNSON,822665.324,32833568.75
"CENTRAL CITY LIQUOR, INC.",POLK,617205.419,23729255.94
HY-VEE #2 (1018) / AMES,STORY,604396.852,17412515.49
HY-VEE #3 FOOD & DRUGSTORE / DAVENPORT,SCOTT,598015.576,19491708.21
HY-VEE WINE AND SPIRITS (1022) / ANKENY,POLK,582990.228,18259640.18
HY-VEE #4 / WDM,POLK,578496.736,18566916.84
HY-VEE / WAUKEE,DALLAS,545560.738,16839781.00
HY-VEE WINE AND SPIRITS / BETTENDORF,SCOTT,543873.158,19103222.99


Crecimeinto Interanual - 2025 vs 2024

In [0]:
WITH VentasCategoriaAnual AS (
  SELECT
    p.category_name,
    SUM(CASE WHEN d.year = 2024 THEN f.sale_dollars ELSE 0 END) AS ventas_2024,
    SUM(CASE WHEN d.year = 2025 THEN f.sale_dollars ELSE 0 END) AS ventas_2025
  FROM iowa_sales.sales_gold.fact_sales f
  JOIN iowa_sales.sales_gold.dim_product p ON f.product_key = p.product_key
  JOIN iowa_sales.sales_gold.dim_date d ON f.date_key = d.date_key
  WHERE d.year IN (2024, 2025)
  GROUP BY p.category_name
)
SELECT
  category_name,
  ventas_2024,
  ventas_2025,
  ROUND(((ventas_2024 - ventas_2025) / NULLIF(ventas_2025, 0)) * 100, 2) AS yoy_growth_pct
FROM VentasCategoriaAnual
WHERE ventas_2024 > 10000 -- Filtramos categorías residuales
ORDER BY yoy_growth_pct DESC
LIMIT 10;

category_name,ventas_2024,ventas_2025,yoy_growth_pct
SPECIAL ORDER ITEMS,270202.89,170243.90,58.72
CORN WHISKIES,118781.08,85046.31,39.67
NEUTRAL GRAIN SPIRITS FLAVORED,3160258.37,2532892.09,24.77
SINGLE MALT SCOTCH,3123931.44,2568561.38,21.62
IMPORTED BRANDIES,8492206.04,7012918.55,21.09
CANADIAN WHISKIES,42322471.49,36046285.99,17.41
AGED DARK RUM,976994.86,857538.35,13.93
FLAVORED RUM,5684051.65,5003428.67,13.60
IRISH WHISKIES,5840754.48,5156079.58,13.28
IMPORTED FLAVORED VODKA,3523406.89,3130799.50,12.54


Rentabilidad real evaluando el costo estatal contra el precio de venta final.

In [0]:
SELECT
  v.vendor_name,
  p.item_description,
  SUM(f.sale_dollars) AS total_revenue,
  SUM(f.state_bottle_cost * f.bottles_sold) AS total_cost,
  SUM(f.sale_dollars) - SUM(f.state_bottle_cost * f.bottles_sold) AS gross_profit_dollars,
  ROUND(((SUM(f.sale_dollars) - SUM(f.state_bottle_cost * f.bottles_sold)) / SUM(f.sale_dollars)) * 100, 2) AS gross_margin_pct
FROM iowa_sales.sales_gold.fact_sales f
JOIN iowa_sales.sales_gold.dim_product p ON f.product_key = p.product_key
JOIN iowa_sales.sales_gold.dim_vendor v ON f.vendor_key = v.vendor_key
GROUP BY v.vendor_name, p.item_description
HAVING total_revenue > 50000 -- Analizar solo productos con un volumen representativo
ORDER BY gross_profit_dollars DESC
LIMIT 15;

vendor_name,item_description,total_revenue,total_cost,gross_profit_dollars,gross_margin_pct
SAZERAC COMPANY INC,FIREBALL CINNAMON WHISKEY,102891304.02,68582591.98,34308712.04,33.34
FIFTH GENERATION INC,TITOS HANDMADE VODKA,97159266.72,64767272.75,32391993.97,33.34
DIAGEO AMERICAS,CAPTAIN MORGAN ORIGINAL SPICED,89871364.85,59893929.93,29977434.92,33.36
DIAGEO AMERICAS,CROWN ROYAL,85592384.46,57049811.42,28542573.04,33.35
BROWN FORMAN CORP.,JACK DANIELS OLD #7 BLACK LABEL,72863163.90,48569150.72,24294013.18,33.34
MOET HENNESSY USA,HENNESSY VS,67936221.04,45280646.14,22655574.90,33.35
CONSTELLATION BRANDS INC,BLACK VELVET,61899948.84,41085708.78,20814240.06,33.63
LUXCO INC,HAWKEYE VODKA,59676583.66,39770648.60,19905935.06,33.36
DIAGEO AMERICAS,CROWN ROYAL REGAL APPLE,58365990.51,38904235.47,19461755.04,33.34
HEAVEN HILL BRANDS,BLACK VELVET,50586534.71,33712136.03,16874398.68,33.36


Elasticidad en Rangos de Precio

In [0]:
SELECT
  CASE 
    WHEN state_bottle_retail < 10 THEN '1. Económico (< \$10)'
    WHEN state_bottle_retail BETWEEN 10 AND 25 THEN '2. Estándar (\$10 - \$25)'
    WHEN state_bottle_retail BETWEEN 25 AND 50 THEN '3. Premium (\$25 - \$50)'
    WHEN state_bottle_retail BETWEEN 50 AND 100 THEN '4. Super Premium (\$50 - \$100)'
    ELSE '5. Lujo (> \$100)'
  END AS price_tier,
  COUNT(DISTINCT sale_invoice_line_no) AS transactions_count,
  SUM(bottles_sold) AS total_bottles_sold,
  SUM(sale_dollars) AS total_revenue
FROM iowa_sales.sales_gold.fact_sales
GROUP BY price_tier
ORDER BY price_tier;

price_tier,transactions_count,total_bottles_sold,total_revenue
1. Económico (< $10),10374131,143176114,742844938.43
2. Estándar ($10 - $25),17510729,136103256,2089829298.04
3. Premium ($25 - $50),4325179,25880734,819818725.95
4. Super Premium ($50 - $100),698858,1830683,115332405.68
5. Lujo (> $100),89255,146735,21972391.54


Concentración de Mercado

In [0]:
WITH TotalVentas AS (
  SELECT SUM(sale_dollars) AS grand_total FROM iowa_sales.sales_gold.fact_sales
)
SELECT
  v.vendor_name,
  SUM(f.sale_dollars) AS revenue,
  ROUND((SUM(f.sale_dollars) / (SELECT grand_total FROM TotalVentas)) * 100, 2) AS market_share_pct
FROM iowa_sales.sales_gold.fact_sales f
JOIN iowa_sales.sales_gold.dim_vendor v ON f.vendor_key = v.vendor_key
GROUP BY v.vendor_name
ORDER BY revenue DESC
LIMIT 10;

vendor_name,revenue,market_share_pct
DIAGEO AMERICAS,747859021.62,19.73
SAZERAC COMPANY INC,403200698.19,10.64
JIM BEAM BRANDS,305635736.22,8.06
PERNOD RICARD USA,240173656.82,6.34
LUXCO INC,211912736.21,5.59
HEAVEN HILL BRANDS,188948338.97,4.99
BACARDI USA INC,185148211.48,4.89
BROWN FORMAN CORP.,173973244.03,4.59
PROXIMO,144646697.70,3.82
CONSTELLATION BRANDS INC,117247062.70,3.09


Historial de Movimientos de Tiendas

In [0]:
SELECT 
  store_business_key AS store_number,
  store_name,
  address,
  county,
  valid_from,
  valid_to,
  is_current
FROM iowa_sales.sales_gold.dim_store
WHERE store_business_key IN (
  -- Subconsulta para encontrar tiendas con más de 1 versión en la dimensión
  SELECT store_business_key
  FROM iowa_sales.sales_gold.dim_store
  GROUP BY store_business_key
  HAVING COUNT(*) > 1
)
ORDER BY store_business_key, valid_from;

store_number,store_name,address,county,valid_from,valid_to,is_current
4046,J AND K MARKET,113 W VANBUREN,APPANOOSE,2026-02-20T05:33:43.196Z,2026-02-21T02:25:26.265Z,false
4046,J AND K MARKET,500 NEW AVENUE,APPANOOSE,2026-02-21T02:25:26.265Z,9999-12-31T00:00:00.000Z,true
4129,CYCLONE LIQUORS,626 LINCOLN WAY,STORY,2026-02-20T05:33:43.196Z,2026-02-21T02:25:26.265Z,false
4129,CYCLONE SUPER LIQUORS,626 LINCOLN WAY,STORY,2026-02-21T02:25:26.265Z,9999-12-31T00:00:00.000Z,true


Categorias con Mayor Volumen de Ventas y Numero de Botellas Total

In [0]:
SELECT
  p.category_name,
  SUM(f.sale_dollars) AS total_sales,
  SUM(f.bottles_sold) AS total_bottles
FROM
  iowa_sales.sales_gold.fact_sales AS f
  JOIN iowa_sales.sales_gold.dim_product AS p ON f.product_key = p.product_key
GROUP BY
  p.category_name
ORDER BY
  total_sales DESC
LIMIT 20;

category_name,total_sales,total_bottles
AMERICAN VODKAS,498845161.38,61682441
CANADIAN WHISKIES,430012187.10,31299502
STRAIGHT BOURBON WHISKIES,274528437.45,14813612
WHISKEY LIQUEUR,226307649.89,34896543
SPICED RUM,193901992.80,14544945
100% AGAVE TEQUILA,174814639.57,6634384
AMERICAN FLAVORED VODKA,137934095.81,12769580
TENNESSEE WHISKIES,133872004.45,6674170
BLENDED WHISKIES,126235172.21,11896332
IMPORTED VODKAS,123654665.69,6730621
